In [1]:
# !pip install scikit-learn mlflow dagshub joblib matplotlib seaborn imbalanced-learn --quiet

import os
import warnings
import joblib
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, classification_report,
    mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
)
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")


In [2]:
from pathlib import Path
ROOT = Path.cwd().parent 
CLEAN_FILE = ROOT / "data" / "cleaned_EMI_dataset.csv"
OUTPUT_DIR = ROOT / "data"

In [3]:
print(f"📥 Loading cleaned dataset: {CLEAN_FILE}")
df = pd.read_csv(CLEAN_FILE)
print(f"Initial shape: {df.shape}")

📥 Loading cleaned dataset: c:\Users\malay chand\Desktop\project\guvi_project\EMIPredict_AI\data\cleaned_EMI_dataset.csv
Initial shape: (404800, 27)


In [4]:
import sys
import os

# Add root folder path (one level up from notebooks/)
root_path = os.path.abspath("..")  
sys.path.append(root_path)

# Add scripts folder explicitly
scripts_path = os.path.join(root_path, "scripts")
sys.path.append(scripts_path)

print(scripts_path)


c:\Users\malay chand\Desktop\project\guvi_project\EMIPredict_AI\scripts


In [5]:
from data_preprocessing import load_and_preprocess_data
from upsampling import apply_smote_upsampling   # 🔥 using your upsampling file

In [6]:
# Classification target
y_class = df['emi_eligibility']

# Regression target
y_reg = df['max_monthly_emi']

# Features
X = df.drop(columns=['emi_eligibility', 'max_monthly_emi'])


In [7]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("Categorical Columns:", categorical_cols)
print("Numerical Columns:", numerical_cols)


Categorical Columns: ['gender', 'marital_status', 'education', 'employment_type', 'company_type', 'house_type', 'existing_loans', 'emi_scenario']
Numerical Columns: ['age', 'monthly_salary', 'years_of_employment', 'monthly_rent', 'family_size', 'dependents', 'school_fees', 'college_fees', 'travel_expenses', 'groceries_utilities', 'other_monthly_expenses', 'current_emi_amount', 'credit_score', 'bank_balance', 'emergency_fund', 'requested_amount', 'requested_tenure']


In [8]:
# Numeric scaling
numeric_transformer = Pipeline([("scaler", StandardScaler())])

# Categorical encoding
categorical_transformer = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

# Combine
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])


In [ ]:
# Encode labels
label_enc = LabelEncoder()
y_class_encoded = label_enc.fit_transform(y_class)

# Apply SMOTE
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(preprocessor.fit_transform(X), y_class_encoded)

# Train-test split
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42, stratify=y_res
)

# Random Forest Classifier
rf_clf = RandomForestClassifier(
    n_estimators=80,
    max_depth=10,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)
rf_clf.fit(X_train_clf, y_train_clf)

# Predictions
y_pred_clf = rf_clf.predict(X_test_clf)

# Metrics
print("Classification Accuracy:", accuracy_score(y_test_clf, y_pred_clf))
print("Classification F1 Score:", f1_score(y_test_clf, y_pred_clf, average='weighted'))
print("\nClassification Report:\n", classification_report(y_test_clf, y_pred_clf))


Classification Accuracy: 0.9757192855354488
Classification F1 Score: 0.9757029576089316

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.97      0.97     62573
           1       0.97      0.99      0.98     62574
           2       0.99      0.96      0.97     62574

    accuracy                           0.98    187721
   macro avg       0.98      0.98      0.98    187721
weighted avg       0.98      0.98      0.98    187721



In [ ]:
# Train-test split
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)

# Random Forest Regressor pipeline
rf_reg = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=80,
        max_depth=12,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

# Train
rf_reg.fit(X_train_reg, y_train_reg)

# Predict
y_pred_reg = rf_reg.predict(X_test_reg)

# Metrics
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
mae = mean_absolute_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)
mape = mean_absolute_percentage_error(y_test_reg, y_pred_reg)

print(f"Regression Metrics:\n RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.4f}, MAPE: {mape:.2f}")


In [ ]:
os.makedirs("models", exist_ok=True)
joblib.dump(rf_clf, "models/rf_classifier.pkl")
joblib.dump(rf_reg, "models/rf_regressor.pkl")
print("✅ Models saved in 'models/' folder")


In [ ]:
mlflow.set_experiment("EMI_Prediction_RandomForest")

with mlflow.start_run(run_name="rf_classification_regression"):
    
    # Classification metrics
    mlflow.log_metric("clf_accuracy", accuracy_score(y_test_clf, y_pred_clf))
    mlflow.log_metric("clf_f1", f1_score(y_test_clf, y_pred_clf, average='weighted'))
    
    # Regression metrics
    mlflow.log_metric("reg_rmse", rmse)
    mlflow.log_metric("reg_mae", mae)
    mlflow.log_metric("reg_r2", r2)
    mlflow.log_metric("reg_mape", mape)
    
    # Log artifacts
    mlflow.log_artifact("models/rf_classifier.pkl")
    mlflow.log_artifact("models/rf_regressor.pkl")
